# Experimento — FolhaUOL (Colab)

Adaptador fino que chama `scripts/experimentar.py` em um cenário `(familia × recorte × protocolo × regime)` da ADR 009 §D.5. **Nenhuma lógica de treino/avaliação vive neste notebook** (CLAUDE.md §5).

Pré-requisitos:
- Ter rodado `preprocessar.ipynb` antes (parquets em `DIR_DADOS_PROC`).
- GPU: **T4** basta para LogReg/SVM/BERTimbau; **L4 obrigatória** para Llama 3.1 8B zero-shot (ciclo 2026-04-24 Q4).
- Runtime → Change runtime type → T4 ou L4.

## 1. Parâmetros do cenário

In [ ]:
# Cenário
FAMILIA = 'logreg'            # logreg | svm | bertimbau | llama31-8b-zs
RECORTE = 'opcao7'            # opcao7 | opcao4 | opcao3
PROTOCOLO = 'rolling'         # rolling | kfold
REGIME = 'multi'              # multi | bin

# Herança de HPs (regra ii da ADR 009 §D.2). None para cenário primário.
# Ex.: '/content/drive/MyDrive/ptbr-market-classification/artifacts/experimentos/20260424-1400-logreg-opcao7-rolling-multi'
HP_DE = None

# Smoke validation em subset estratificado por mês (None = corpus completo).
SMOKE_N = None

# Repositório e Drive
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification-2.git'  # substituir pela URL do seu fork
RAMO = 'master'
DIR_REPO = '/content/ptbr-market-classification-2'
DIR_DRIVE = '/content/drive/MyDrive/ptbr-market-classification-2'
DIR_DADOS_PROC = f'{DIR_DRIVE}/data/processado'
DIR_ARTEFATOS_EXP = f'{DIR_DRIVE}/artifacts/experimentos'

## 2. Montar Drive e preparar repositório

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

if os.path.exists(DIR_REPO):
    subprocess.run(['git', '-C', DIR_REPO, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'checkout', RAMO], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', RAMO, REPO_URL, DIR_REPO], check=True)

os.chdir(DIR_REPO)
print('cwd =', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## 3. Verificar GPU (obrigatória para BERTimbau/Llama)

In [ ]:
import subprocess

try:
    saida = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print(saida.stdout.strip() or 'nenhuma GPU visível')
except FileNotFoundError:
    print('nvidia-smi não encontrado — runtime sem GPU')

if FAMILIA in ('bertimbau', 'llama31-8b-zs'):
    import torch
    assert torch.cuda.is_available(), f'Família {FAMILIA} requer GPU; troque o runtime.'
    nome_gpu = torch.cuda.get_device_name(0)
    print(f'GPU ativa: {nome_gpu}')
    if FAMILIA == 'llama31-8b-zs' and 'L4' not in nome_gpu and 'A100' not in nome_gpu:
        print(f'AVISO: Llama 3.1 8B foi validada em L4 (Q4). GPU atual: {nome_gpu}.')

## 4. Verificar parquets pré-processados

In [ ]:
from pathlib import Path

parquet = Path(DIR_DADOS_PROC) / f'corpus_{RECORTE}.parquet'
assert parquet.exists(), (
    f'Parquet ausente: {parquet}. Rode preprocessar.ipynb antes.'
)
print(f'Corpus: {parquet} ({parquet.stat().st_size / 1024**2:.1f} MB)')

## 5. Executar cenário

In [ ]:
import os
os.environ['PTBR_MC_DIR_DADOS_PROC'] = DIR_DADOS_PROC
os.environ['PTBR_MC_DIR_EXP'] = DIR_ARTEFATOS_EXP

comando = ['python', 'scripts/experimentar.py',
           '--familia', FAMILIA,
           '--recorte', RECORTE,
           '--protocolo', PROTOCOLO,
           '--regime', REGIME]
if HP_DE:
    comando += ['--hp-de', HP_DE]
if SMOKE_N:
    comando += ['--smoke', str(SMOKE_N)]

print('$ ' + ' '.join(comando))
!{' '.join(comando)}

## 6. Resumo do experimento

In [ ]:
import json
from pathlib import Path

raiz = Path(DIR_ARTEFATOS_EXP)
padrao = f'*-{FAMILIA}-{RECORTE}-{PROTOCOLO}-{REGIME}*'
candidatos = sorted(raiz.glob(padrao), key=lambda p: p.stat().st_mtime)
assert candidatos, f'Nenhum experimento encontrado para {padrao}'
dir_exp = candidatos[-1]
print(f'Último experimento: {dir_exp.name}\n')

summary = json.loads((dir_exp / 'summary.json').read_text())
print('Métricas cross-fold:')
for chave in ('f1_binario_projetado', 'pr_auc_binario_projetado', 'macro_f1'):
    v = summary.get(chave, {})
    if v and v.get('media') is not None:
        print(f'  {chave}: {v["media"]:.4f}  IC95%=[{v["ic_95_inf"]:.4f}, {v["ic_95_sup"]:.4f}]  dp={v["dp"]:.4f}')

print()
print('Por fold:')
for f in summary['por_fold']:
    pra = f.get('pr_auc_binario_projetado')
    pra_s = f'{pra:.4f}' if pra is not None else 'NA'
    print(f'  fold {f["fold"]}: F1bin={f["f1_binario_projetado"]:.4f}  PR-AUCbin={pra_s}  macroF1={f["macro_f1"]:.4f}  n_teste={f["n_teste"]}')

leak_path = dir_exp / 'leakage_diagnostic.json'
if leak_path.exists():
    leak = json.loads(leak_path.read_text())
    print(f'\nLeakage agregado: {leak["agregado"]}')